# GraphRAG Playground: del dato a la respuesta

Un recorrido corto por la aplicación. Cada bloque tiene un objetivo. El código y sus comentarios están en inglés; las explicaciones están en español.

**Flujo:** datos → validación → Neo4j → recuperación → LangGraph → respuesta → MLflow.

Las llamadas de este notebook usan servicios reales una vez configurados. No contiene claves ni celdas que borren bases o endpoints.

## 1. Preparar el entorno
Instala `requirements-dev.txt` desde la raíz del repositorio y selecciona ese entorno como kernel. El archivo `.env` se mantiene fuera de Git.

In [ ]:
# Locate the repository when the notebook is opened from its own folder.
from pathlib import Path
import os
import sys

project_dir = Path.cwd()
if not (project_dir / "models.py").exists():
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))
os.chdir(project_dir)

from dotenv import load_dotenv
load_dotenv(project_dir / ".env")
print("Configuration loaded. Secret values are not displayed.")

## 2. Ver los datos
Cada entidad tiene nombre y tipo. Cada relación tiene origen, tipo y destino. El ejemplo usa ocho entidades y ocho relaciones; se puede editar.

In [ ]:
# Inspect a small dataset before sending it to any service.
import pandas as pd
from sample_data import SAMPLE_GRAPH

display(pd.DataFrame([entity.model_dump() for entity in SAMPLE_GRAPH.entities]))
display(pd.DataFrame([relation.model_dump() for relation in SAMPLE_GRAPH.relations]))

## 3. Validar antes de escribir
`MovieGraph` comprueba los límites, nombres únicos y tipos de cada conexión. Por ejemplo, `ACTED_IN` conecta Person → Movie. Esta validación es código determinista.

In [ ]:
# Validation runs before a database transaction starts.
from models import MovieGraph

graph_data = MovieGraph.model_validate(SAMPLE_GRAPH.model_dump())
print(len(graph_data.entities), "entities")
print(len(graph_data.relations), "relationships")

## 4. Conectar Neo4j
El driver usa las credenciales del servidor. `verify()` comprueba acceso sin crear datos. La aplicación guarda sus propios nodos con la etiqueta `GRPlayEntity`.

In [ ]:
# Open the driver and check connectivity without modifying data.
from graph_store import Neo4jStore

store = Neo4jStore(
    uri=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database=os.getenv("NEO4J_DATABASE", "neo4j"),
)
store.verify()
print("Neo4j connection verified.")

## 5. Activar MLflow
El destino puede ser Databricks o una base SQLite local. El autolog captura LangChain/LangGraph; las funciones de guardado y recuperación añaden sus propias trazas. Una traza registra lo ocurrido, no demuestra que la respuesta sea correcta.

In [ ]:
# Configure the tracking destination and automatic LangChain traces.
from observability import configure_tracking

experiment_id = configure_tracking()
print("MLflow experiment:", experiment_id)

## 6. Guardar un grafo aislado
Esta celda sí crea un snapshot de nuestros datos. Devuelve un identificador aleatorio. Todas las consultas usarán ese identificador; no modificamos grafos anteriores.

In [ ]:
# Save a new snapshot and record the real operation in MLflow.
import uuid
from observability import save_graph_with_tracking

from usage_limits import UsageBudget

budget = UsageBudget(store)
session_id = uuid.uuid4().hex
budget.claim("save", session_id)
store.cleanup_expired()
graph_id = save_graph_with_tracking(store, graph_data, session_id)
print("Graph saved:", graph_id)

## 7. Entender la recuperación sin LLM
Primero construimos un plan explícito: buscar el reparto de Inception. La aplicación selecciona una consulta Cypher que ya está escrita, con parámetros. El LLM no puede escribir ni ejecutar Cypher arbitrario.

In [ ]:
# Query the database directly to inspect the retrieved evidence.
from models import QueryPlan
from graph_store import READ_QUERIES

plan = QueryPlan(intent="movie_cast", first_name="Inception", second_name="")
print(READ_QUERIES[plan.intent])
facts = store.retrieve(graph_id, plan)
pd.DataFrame(facts)

## 8. Crear el LLM
Groq interpreta la pregunta y redacta la respuesta. El esquema `QueryPlan` limita las intenciones disponibles. Aun así, validamos los nombres y tipos que devuelve el modelo.

In [ ]:
# Prepare structured model outputs; this cell does not make a model call yet.
from workflow import GroqResponder

responder = GroqResponder(
    api_key=os.environ["GROQ_API_KEY"],
    model=os.getenv("GROQ_MODEL", "openai/gpt-oss-120b"),
    budget=budget,
)

## 9. Conectar las etapas con LangGraph
LangGraph controla el flujo. Neo4j almacena las relaciones. Son dos grafos distintos. Si no recuperamos hechos, la ruta termina con una abstención.

In [ ]:
# Build and inspect the workflow without executing a question.
from workflow import build_workflow

workflow = build_workflow(store, responder)
print(workflow.get_graph().draw_mermaid())

## 10. Hacer una pregunta completa
Esta celda llama realmente a Groq, consulta Neo4j y escribe en MLflow. La tabla permite contrastar la respuesta con los hechos recuperados.

In [ ]:
# Run one end-to-end question and persist its trace and metrics.
from observability import ask_with_tracking

budget.claim("question", session_id)
result = ask_with_tracking(
    workflow=workflow,
    question="Which actors are shared by Inception and The Dark Knight Rises?",
    graph_id=graph_id,
    entities=[entity.model_dump() for entity in graph_data.entities],
    session_id=session_id,
)
print(result["answer"])
print("Status:", result["status"])
print("Trace:", result["trace_id"])
pd.DataFrame(result["facts"])

## 11. Inspeccionar operaciones reales
Consultamos el histórico persistido de esta sesión. Duración y errores son métricas operativas. La calidad requiere preguntas con respuestas esperadas; tokens y coste dependen de la información del proveedor.

In [ ]:
# Read the recorded operations rather than generating demonstration metrics.
from observability import read_operations

operations = read_operations(experiment_id, session_id)
columns = ["start_time", "tags.operation", "tags.result", "metrics.duration_seconds", "metrics.fact_count"]
operations[[column for column in columns if column in operations]]

## 12. Comprobar una abstención
Titanic no está en el snapshot. La aplicación debe reconocer el límite de los datos, aunque el modelo conozca la película por su entrenamiento.

In [ ]:
# Test a question about an entity outside this graph.
budget.claim("question", session_id)
missing = ask_with_tracking(
    workflow, "Who acted in Titanic?", graph_id,
    [entity.model_dump() for entity in graph_data.entities], session_id,
)
print(missing["status"])
print(missing["answer"])

## 13. Cerrar la conexión
Esto cierra el driver local; no borra el grafo ni las métricas.

In [ ]:
# Release the local pool. Graphs expire after 24 hours; MLflow history remains separate.
store.close()

## Preguntas para defender el proyecto

1. ¿Por qué una búsqueda de actores compartidos se expresa bien con relaciones?
2. ¿Qué hace LangGraph que no hace Neo4j?
3. ¿Cómo impedimos consultar el grafo de otra sesión?
4. ¿Qué demuestra una traza y qué no demuestra?
5. ¿Cómo detectarías una respuesta incorrecta con citas válidas?
6. ¿Qué cambia al pasar de pruebas propias a usuarios reales?

**Atribución:** implementación propia con asistencia de IA; la secuencia didáctica se inspira en el tutorial GraphRAG de DSwithBappy enlazado en el README. No se ha copiado su notebook.

## Límites y limpieza de la demo pública
10 preguntas por sesión; cada una hace hasta dos llamadas. La cuota global sigue Groq (30 llamadas/minuto, 1.000/día, 8.000 tokens/minuto, 200.000/día), sin tope mensual. `UsageBudget` reserva en una transacción Neo4j antes de llamar al modelo y ajusta con el uso real después. Un reinicio no borra los contadores. Los grafos caducan a las 24 h; se limpian al guardar o preguntar. Las trazas en Databricks se conservan por separado.
